# VietDub native vocoder — Kaggle GPU training

**Research workspace** for the independently trained VietDub HiFi-GAN-inspired vocoder. It reconstructs recorded Vietnamese speech from mel features; **it does not synthesize text**. No external TTS checkpoints are loaded.

Attach an authorized **Mozilla Common Voice Vietnamese** dataset via Kaggle `Add Input`, preserving `validated.tsv` and `clips/`. The official catalog lists this corpus as CC0, but you must review the precise release, usage conditions and voice rights. The default approval toggles below are deliberately `False`.

This notebook has not been executed on Kaggle; evaluate GPU runtime and costs before long runs. Do **not** expose original clips, trained checkpoints or Kaggle secrets publicly.


In [ ]:
from pathlib import Path
import os

REPO = Path('/kaggle/working/vietdub-tts')
SOURCE_ROOT = Path('/kaggle/input/common-voice-vi')  # Change to your authorized mounted input
DATASET_WORKSPACE = Path('/kaggle/working/vietdub-corpus')
TRAIN_OUTPUT = Path('/kaggle/working/vietdub-vocoder-run')

RIGHTS_REVIEWED = False          # Set True only after human review of the exact release.
APPROVE_VOCODER_TRAINING = False # Set True only after authorizing voice training.

TRAIN_STEPS = 100000             # Total target steps, NOT extra steps when resuming
CROP_FRAMES = 48
CHECKPOINT_EVERY = 250

assert SOURCE_ROOT.is_dir(), f'Attach a licensed Common Voice input folder: {SOURCE_ROOT}'
assert (SOURCE_ROOT/'validated.tsv').is_file() and (SOURCE_ROOT/'clips').is_dir(), 'Expected validated.tsv + clips/'
assert RIGHTS_REVIEWED and APPROVE_VOCODER_TRAINING, 'Rights and voice-training authorization require explicit review'


In [ ]:
import subprocess
import sys

if not REPO.is_dir():
    subprocess.run(['git','clone','--depth','1','--branch','feat/native-hifigan-vocoder-superpowers',
                    'https://github.com/hungtvb/vietdub-tts.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
import torch
print('GPU available:',torch.cuda.is_available(), 'device:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable GPU accelerator in Kaggle notebook settings'


In [ ]:
from vietdub_native.commonvoice_vocoder import prepare_commonvoice
import json

if not (DATASET_WORKSPACE/'vocoder.jsonl').is_file():
    report = prepare_commonvoice(SOURCE_ROOT, DATASET_WORKSPACE, max_clips=10000,
                                 rights_reviewed=RIGHTS_REVIEWED,
                                 training_approved=APPROVE_VOCODER_TRAINING)
else:
    report = json.loads((DATASET_WORKSPACE/'import_report.json').read_text())
print(json.dumps(report,indent=2))
assert report['train_clips'] >= 2 and report['dev_clips'] >= 1


In [ ]:
from vietdub_native.hifigan_train import train_hifigan

resume = (TRAIN_OUTPUT/'private_checkpoints'/'training_state.pt').is_file()
report = train_hifigan(DATASET_WORKSPACE/'vocoder.jsonl', TRAIN_OUTPUT,
                       steps=TRAIN_STEPS, crop_frames=CROP_FRAMES,
                       device='cuda', resume=resume,
                       checkpoint_every=CHECKPOINT_EVERY,
                       eval_seconds=5)
print(json.dumps(report,indent=2))
assert report['intelligibility_verified'] is False


In [ ]:
from IPython.display import display, Audio

# This is generated reconstruction, NOT the original audio and NOT text-to-speech.
output_wav = TRAIN_OUTPUT/'dev_reconstructed.wav'
assert output_wav.exists()
display(Audio(filename=str(output_wav)))
print('Listen next to your privately licensed held-out original; do not infer success from loss/CI alone.')
print('Persist private_checkpoints/training_state.pt securely to resume after Kaggle session ends.')
